# Player clusters & look-alikes

Two similarity spaces exist on disk — **baseline cosine** (66 residualized choice features)
and **Player Vectors** (20 NMF weights over location heatmaps) — plus the **quality axis**
(npxG+xA/90, separate table by design). This notebook combines them:

- clusters: browse the KMeans role groups (side-specific: L/R backs and wingers separate)
- look-alikes: pairs close in *both* spaces with a small quality gap —
  players you'd expect to **yield similar output**
- `lookalikes('name')`: the query version, quality shown next to style, never mixed into it

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'config.yaml').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src' / 'models'))
from trajectories import load_vectors, feature_cols

KEYS = ['player_id', 'team_id', 'season']
vec = load_vectors(ROOT)
F = feature_cols(vec)
pv = pd.read_parquet(ROOT / 'data/models/player_vectors.parquet')
quality = pd.read_parquet(ROOT / 'data/features/player_season_quality.parquet',
                          columns=KEYS + ['nonpenalty_xg_plus_xa_per_90'])

# one aligned frame: solid entities, both vector spaces, cluster, quality
df = (vec[vec.minutes >= 450]
      .merge(pv[KEYS + ['cluster'] + [c for c in pv.columns if '_c' in c]], on=KEYS)
      .merge(quality, on=KEYS, how='left')
      .reset_index(drop=True))
PVDIMS = [c for c in pv.columns if '_c' in c]
print(f'{len(df):,} entities | {len(F)} style dims | {len(PVDIMS)} PV dims')

## 1 · Clusters — the role groups

KMeans on standardized PV weights. Pick a cluster, browse its members.

In [ ]:
summary = (df.groupby('cluster')
             .agg(n=('player', 'size'),
                  top_pos=('pos_group', lambda s: s.value_counts().idxmax()),
                  pos_share=('pos_group', lambda s: s.value_counts(normalize=True).iloc[0]),
                  minutes_med=('minutes', 'median')))
summary

In [ ]:
def cluster_members(c, n=15):
    cols = ['player', 'team', 'season', 'context', 'pos_group', 'minutes',
            'nonpenalty_xg_plus_xa_per_90']
    return df[df.cluster == c].nlargest(n, 'minutes')[cols]

cluster_members(5)          # right-side attackers; try 7 (left), 2/4 (RB/LB), 6 (GK)

## 2 · Look-alikes — similar style in both spaces, similar output

Rank every distinct-player pair by agreement between the two representations, then keep
pairs whose quality gap is small. Both spaces agreeing kills each one's blind spot:
cosine can't see volume, Player Vectors can't see choices. GKs excluded (their vectors
are all near-identical and would own the list); 900+ min for stability.

In [ ]:
sub = df[(df.minutes >= 900) & (df.pos_group != 'GK')].reset_index(drop=True)

Z1 = sub[F].to_numpy(float)
Z1 = Z1 / np.linalg.norm(Z1, axis=1, keepdims=True)
S_cos = Z1 @ Z1.T                                   # style cosine
V2 = sub[PVDIMS].to_numpy(float)
sq = (V2 ** 2).sum(1)
D_pv = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * V2 @ V2.T, 0))  # PV euclidean

iu = np.triu_indices(len(sub), k=1)
pairs = pd.DataFrame({'i': iu[0], 'j': iu[1],
                      'cos': S_cos[iu], 'pv_dist': D_pv[iu]})
pairs = pairs[sub.player_id.values[pairs.i] != sub.player_id.values[pairs.j]]
# agreement score: mean percentile across the two spaces (higher = more alike)
pairs['score'] = (pairs.cos.rank(pct=True) + (-pairs.pv_dist).rank(pct=True)) / 2
q = sub.nonpenalty_xg_plus_xa_per_90.values
pairs['q_gap'] = np.abs(q[pairs.i] - q[pairs.j])
print(f'{len(pairs):,} distinct-player pairs scored')

In [ ]:
def show(p, n=12):
    a, b = sub.iloc[p.i.values], sub.iloc[p.j.values]
    return pd.DataFrame({
        'player_a': a.player.values, 'a': a.team.values + ' ' + a.season.values,
        'player_b': b.player.values, 'b': b.team.values + ' ' + b.season.values,
        'pos': a.pos_group.values, 'cos': p.cos.round(3).values,
        'pv_dist': p.pv_dist.round(2).values, 'q_gap': p.q_gap.round(2).values,
    }).head(n)

# output twins: top agreement AND quality gap under 0.1 npxG+xA/90
twins = pairs[pairs.q_gap < 0.10].sort_values('score', ascending=False)
show(twins)

In [ ]:
# per-position view — the same list, best pairs within each role
top_by_pos = (twins.assign(pos=sub.pos_group.values[twins.i])
              .groupby('pos', group_keys=False).head(4)
              .sort_values(['pos', 'score'], ascending=[True, False]))
show(top_by_pos, n=16)

## 3 · Query: look-alikes for one player

Style picks the neighbours; quality is a *column*, never part of the distance
(PLAN §5.2: closest in style, then ranked by quality).

In [ ]:
def lookalikes(name, season=None, k=12):
    m = sub[sub.player.str.contains(name, case=False, na=False)]
    if season is not None:
        m = m[m.season == season]
    qi = m.minutes.idxmax()
    qq = sub.loc[qi]
    print(f'query: {qq.player} — {qq.team} {qq.season} ({qq.pos_group}, '
          f'npxG+xA/90 {qq.nonpenalty_xg_plus_xa_per_90:.2f}, cluster {qq.cluster})')
    d = sub.assign(cos=S_cos[qi], pv_dist=D_pv[qi]).drop(index=qi)
    d = d[(d.pos_group == qq.pos_group) & (d.player_id != qq.player_id)]
    d['score'] = (d.cos.rank(pct=True) + (-d.pv_dist).rank(pct=True)) / 2
    d['same_cluster'] = d.cluster == qq.cluster
    cols = ['player', 'team', 'season', 'context', 'same_cluster',
            'cos', 'pv_dist', 'nonpenalty_xg_plus_xa_per_90', 'minutes']
    return d.nlargest(k, 'score')[cols].round(3)

lookalikes('Griezmann')

## 4 · Notes

- Both-space agreement is strict: it demands same choices **and** same locations **and**
  similar volume. Disagreements between the spaces are informative (mirror-image wingers
  agree in choices, never in location).
- 2015/16 league entities dominate — that's corpus composition (the only all-teams men's
  league season), not a similarity fact.
- Candidate pairs for the E6 human-triplet study and E5 sanity checks come straight from
  the `twins` table.